# Análisis principal del TFG

En este notebook desarrollo el análisis empírico principal del trabajo. Parto de la base `df_total_preparado.csv`, que ya queda construida en el notebook descriptivo/exploratorio. De esta forma, aquí no repito la limpieza ni la recodificación de variables, sino que me concentro directamente en estimar e interpretar los resultados.

El objetivo del análisis es comprobar si el shock religioso externo de abril-mayo de 2025 se refleja primero en la religiosidad declarada o en la práctica religiosa y, a partir de ahí, si aparecen señales compatibles con una traslación hacia variables políticas o de percepción de los problemas en España. No se trata de volver a mostrar que existen diferencias religiosas entre partidos, porque eso ya se ha visto en el análisis descriptivo, sino de estudiar si esas variables se mueven de forma relevante durante la ventana del shock.

La interpretación tiene que ser prudente. Los datos proceden de barómetros mensuales distintos, no de un panel de las mismas personas. Por eso, los resultados se leen como cambios agregados compatibles con el shock, no como una prueba cerrada de que cada individuo haya cambiado su voto o su religiosidad.


## Carga de la base preparada

Cargo la base ya preparada en el notebook descriptivo. Esta base incluye la ideología en formato numérico, el identificador único, las variables `religiosidad_si_no` y `practica_religion_si_no`, las variables binarias de voto y el criterio de ponderación con `Peso` ya fijado.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_data_path(filename="df_total_preparado.csv"):
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidate = folder / "data" / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"No encuentro data/{filename} desde {Path.cwd()}. "
        "Ejecuta antes el notebook descriptivo/exploratorio para guardar la base preparada."
    )


def media_ponderada(serie, pesos):
    return np.average(serie, weights=pesos)


DATA_PATH = find_data_path()
df_total = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print(f"Base cargada: {DATA_PATH}")
print(df_total.shape)
df_total.head()


Base cargada: /Users/tomi/Desktop/TFG - ECO/data/df_total_preparado.csv
(20075, 32)


,Mes/Año,id,CCAA,Poblacion_municipio,Sexo,Edad,Ideologia,Religiosidad,Practica_religion,Estudios,...,Recuerdo_voto_2023,Problema_1,Problema_2,Problema_3,religiosidad_si_no,practica_religion_si_no,voto_psoe_si_no,voto_pp_si_no,voto_vox_si_no,voto_sumar_si_no
0,03/2025,03/2025_28203,Andalucía,10.001 a 50.000 habitantes,Hombre,45,5.0,"Indiferente, no creyente",N.P.,Primaria,...,N.P.,El mal comportamiento de los/as políticos/as,"La crisis económica, los problemas de índole e...",La inmigración,0,0.0,0.0,0.0,0.0,0.0
1,03/2025,03/2025_63387,Andalucía,10.001 a 50.000 habitantes,Mujer,60,1.0,Católico/a no practicante,Nunca,Secundaria 1ª etapa,...,PSOE,Los problemas políticos en general,El mal comportamiento de los/as políticos/as,La inmigración,1,0.0,0.0,0.0,0.0,0.0
2,03/2025,03/2025_74875,Andalucía,10.001 a 50.000 habitantes,Hombre,60,3.0,Católico/a no practicante,Casi nunca,Superiores,...,PSOE,"Las desigualdades, incluida la de género, dife...",La vivienda,La sanidad,1,0.0,1.0,0.0,0.0,0.0
3,03/2025,03/2025_10108,Andalucía,10.001 a 50.000 habitantes,Hombre,43,4.0,Católico/a no practicante,Varias veces al año,Secundaria 1ª etapa,...,PSOE,Los problemas relacionados con la juventud. Fa...,La sanidad,La inmigración,1,1.0,1.0,0.0,0.0,0.0
4,03/2025,03/2025_3233,Andalucía,100.001 a 400.000 habitantes,Mujer,38,10.0,Católico/a no practicante,Varias veces al año,Secundaria 1ª etapa,...,PACMA,El Gobierno y partidos o políticos/as concreto...,La inmigración,El paro,1,1.0,0.0,0.0,1.0,0.0


# 1. Regresiones base

En este primer bloque estimo las regresiones principales del trabajo. La lógica es ir de lo más cercano al shock a lo más indirecto: primero religiosidad declarada, después práctica religiosa y, a continuación, intención de voto e ideología. Todas las regresiones mantienen el mismo criterio: marzo de 2025 como referencia, controles de edad, sexo y estudios, ponderación con `Peso` y errores estándar robustos.


## Regresión 1: religiosidad declarada

El primer paso sustantivo es comprobar si el shock aparece en la variable más cercana al acontecimiento: la religiosidad declarada. Si la muerte del papa Francisco y la elección de León XIV aumentaron la saliencia pública de la religión, lo más razonable es buscar primero un cambio en la probabilidad de declararse católico/a.

Para ello estimo una regresión en la que `religiosidad_si_no` depende del mes de encuesta, controlando por edad, sexo y estudios. Marzo de 2025 funciona como referencia previa al shock, abril como mes de transición y mayo-julio como ventana posterior. Mantengo el uso de `Peso`, siguiendo el criterio fijado en el notebook descriptivo.


In [2]:
variables_reg1 = [
    "religiosidad_si_no",
    "Mes/Año",
    "Edad",
    "Sexo",
    "Estudios",
    "Peso",
]

df_reg1 = df_total[variables_reg1].copy()

df_reg1.isna().sum()


religiosidad_si_no     0
Mes/Año                0
Edad                   0
Sexo                   0
Estudios              18
Peso                   0
dtype: int64

Antes de estimar, reviso qué observaciones se pierden por falta de información en `Estudios`, porque esta variable entra como control en el modelo.


In [3]:
df_reg1[df_reg1["Estudios"].isna()]

,religiosidad_si_no,Mes/Año,Edad,Sexo,Estudios,Peso
6121,0,04/2025,84,Hombre,NaN,5.07996
6824,1,04/2025,79,Mujer,NaN,4.45785
7277,0,04/2025,54,Mujer,NaN,3.00145
8709,1,05/2025,45,Hombre,NaN,3.60887
9442,0,05/2025,47,Hombre,NaN,3.46510
10479,1,05/2025,63,Hombre,NaN,4.27647
11092,1,05/2025,74,Hombre,NaN,2.66302
11206,1,05/2025,89,Hombre,NaN,3.36553
12303,1,06/2025,72,Hombre,NaN,2.30078
13826,0,06/2025,49,Hombre,NaN,4.27672


Elimino las observaciones incompletas para que la regresión trabaje con una muestra comparable en todas las variables utilizadas.


In [4]:
df_reg1 = df_reg1.dropna().copy()

Compruebo el tamaño final de la muestra. También reviso la distribución por mes, sexo y estudios para asegurar que no se ha producido una pérdida extraña de observaciones.


In [5]:
df_reg1.shape

(20057, 6)

Primero miro la distribución básica de la muestra preparada para esta regresión.


In [6]:
df_reg1["Mes/Año"].value_counts().sort_index()
df_reg1["Sexo"].value_counts()
df_reg1["Estudios"].value_counts()

Estudios
Superiores             10453
F.P.                    3999
Secundaria 2ª etapa     2794
Secundaria 1ª etapa     2097
Primaria                 503
Sin estudios             205
Otros                      6
Name: count, dtype: int64

Muestro esas distribuciones de forma más clara antes de estimar el modelo.


In [7]:
display(df_reg1["Mes/Año"].value_counts().sort_index())
display(df_reg1["Sexo"].value_counts())
display(df_reg1["Estudios"].value_counts())

Mes/Año
03/2025    4018
04/2025    4005
05/2025    4013
06/2025    4008
07/2025    4013
Name: count, dtype: int64

Sexo
Hombre    10550
Mujer      9507
Name: count, dtype: int64

Estudios
Superiores             10453
F.P.                    3999
Secundaria 2ª etapa     2794
Secundaria 1ª etapa     2097
Primaria                 503
Sin estudios             205
Otros                      6
Name: count, dtype: int64

### Funciones auxiliares para el análisis principal

Antes de estimar la primera regresión, dejo preparadas algunas funciones comunes. La idea es que todos los modelos base sigan la misma estructura: efectos de mes, controles de edad, sexo y estudios, ponderación con `Peso` y errores estándar robustos HC1. Esto ayuda a que los resultados sean comparables entre sí.


In [8]:
import statsmodels.formula.api as smf


MESES_COMPARACION = ["04/2025", "05/2025", "06/2025", "07/2025"]
FORMULA_CONTROLES_BASE = 'C(Q("Mes/Año")) + Edad + C(Sexo) + C(Estudios)'


def construir_formula_base(variable_dependiente):
    return f"{variable_dependiente} ~ {FORMULA_CONTROLES_BASE}"


def estimar_wls_base(df, variable_dependiente, peso="Peso"):
    return smf.wls(
        formula=construir_formula_base(variable_dependiente),
        data=df,
        weights=df[peso],
    ).fit(cov_type="HC1")


def tabla_resumen_dependiente(df, variable, peso="Peso"):
    filas = []
    for mes, grupo in df.groupby("Mes/Año", sort=True):
        filas.append({
            "Mes/Año": mes,
            "n": len(grupo),
            "peso_total": grupo[peso].sum(),
            "media_ponderada": media_ponderada(grupo[variable], grupo[peso]),
        })
    return pd.DataFrame(filas).round(4)


def estrellas_significatividad(p_value):
    if pd.isna(p_value):
        return ""
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.1:
        return "*"
    return ""


def formatear_coeficiente(coeficiente, p_value):
    return f"{coeficiente:.4f}{estrellas_significatividad(p_value)}"


def extraer_coeficientes_meses(modelo, regresion, variable):
    intervalo_confianza = modelo.conf_int()
    filas = []

    for mes in MESES_COMPARACION:
        termino = f'C(Q("Mes/Año"))[T.{mes}]'
        if termino not in modelo.params.index:
            continue

        filas.append({
            "regresion": regresion,
            "variable": variable,
            "mes": mes,
            "coeficiente": modelo.params[termino],
            "p_value": modelo.pvalues[termino],
            "ic_95_inf": intervalo_confianza.loc[termino, 0],
            "ic_95_sup": intervalo_confianza.loc[termino, 1],
        })

    return pd.DataFrame(filas)


formula_reg1 = construir_formula_base("religiosidad_si_no")
modelo_reg1 = estimar_wls_base(df_reg1, "religiosidad_si_no")


Estimo el modelo y miro dos cosas: la media ponderada por mes, para tener la evolución bruta, y el resumen de la regresión, para ver si las diferencias frente a marzo se mantienen al introducir controles.


In [9]:
display(tabla_resumen_dependiente(df_reg1, "religiosidad_si_no"))
modelo_reg1.summary()


,Mes/Año,n,peso_total,media_ponderada
0,03/2025,4018,4017.9960,0.5445
1,04/2025,4005,3995.4582,0.5552
2,05/2025,4013,4000.6174,0.5850
3,06/2025,4008,3997.5342,0.5612
4,07/2025,4013,3997.7997,0.5616


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:     religiosidad_si_no   R-squared:                       0.071
Model:                            WLS   Adj. R-squared:                  0.071
Method:                 Least Squares   F-statistic:                     73.47
Date:                Thu, 21 May 2026   Prob (F-statistic):          5.08e-177
Time:                        14:07:01   Log-Likelihood:                -16918.
No. Observations:               20057   AIC:                         3.386e+04
Df Residuals:                   20044   BIC:                         3.396e+04
Df Model:                          12                                         
Covariance Type:                  HC1                                         
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              0.2179      0.020     10.781      0.000       0.178       0.258
C(Q("Mes/Año"))[T.04/2025]             0.0116      0.015      0.786      0.432      -0.017       0.041
C(Q("Mes/Año"))[T.05/2025]             0.0410      0.015      2.809      0.005       0.012       0.070
C(Q("Mes/Año"))[T.06/2025]             0.0168      0.015      1.123      0.261      -0.013       0.046
C(Q("Mes/Año"))[T.07/2025]             0.0187      0.015      1.206      0.228      -0.012       0.049
C(Sexo)[T.Mujer]                       0.0602      0.010      6.243      0.000       0.041       0.079
C(Estudios)[T.Otros]                   0.3039      0.162      1.879      0.060      -0.013       0.621
C(Estudios)[T.Primaria]                0.0732      0.024      3.033      0.002       0.026       0.121
C(Estudios)[T.Secundaria 1ª etapa]     0.0544      0.015      3.530      0.000       0.024       0.085
C(Estudios)[T.Secundaria 2ª etapa]    -0.0308      0.014     -2.250      0.024      -0.058      -0.004
C(Estudios)[T.Sin estudios]            0.0098      0.035      0.284      0.776      -0.058       0.078
C(Estudios)[T.Superiores]             -0.0582      0.011     -5.518      0.000      -0.079      -0.038
Edad                                   0.0057      0.000     17.826      0.000       0.005       0.006
==============================================================================
Omnibus:                      448.566   Durbin-Watson:                   1.988
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              459.717
Skew:                          -0.353   Prob(JB):                    1.49e-100
Kurtosis:                       2.771   Cond. No.                     1.41e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
[2] The condition number is large, 1.41e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Conclusiones de la regresión 1:


Mayo es el único mes en el que aparece un aumento claro de la religiosidad declarada. El coeficiente es `+0,0410`, lo que equivale a algo más de 4 puntos porcentuales respecto a marzo, una vez controladas edad, sexo y estudios.

Abril no muestra un cambio claro, algo razonable porque el fallecimiento de Francisco se produce al final del mes. Junio y julio siguen con coeficientes positivos, pero más pequeños y sin la misma claridad estadística.


Este resultado encaja con la lógica del trabajo: la primera señal del shock aparece justo en la variable más próxima al hecho religioso y en el mes de mayor saliencia pública, mayo. No implica por sí solo que haya cambiado el voto, pero sí ofrece una base para seguir preguntando si esa activación religiosa tuvo alguna proyección política.


## Regresión 2: práctica religiosa

Después de la religiosidad declarada, analizo la práctica religiosa. Esta variable ya viene construida en la base preparada, después de tratar el problema de los `NaN` de mayo y de separar quienes practican al menos varias veces al año de quienes no practican o no procede.

La pregunta aquí es si el shock se refleja también en comportamiento religioso, no solo en identificación religiosa. En principio, cabe esperar que la práctica sea más estable que la identidad declarada, porque depende más de hábitos previos.


In [10]:
variables_reg2 = [
    "practica_religion_si_no",
    "Mes/Año",
    "Edad",
    "Sexo",
    "Estudios",
    "Peso",
]

df_reg2 = df_total[variables_reg2].dropna().copy()
df_reg2["practica_religion_si_no"] = df_reg2["practica_religion_si_no"].astype(int)

display(df_reg2.isna().sum())
display(df_reg2.shape)
display(df_reg2["Mes/Año"].value_counts().sort_index())


practica_religion_si_no    0
Mes/Año                    0
Edad                       0
Sexo                       0
Estudios                   0
Peso                       0
dtype: int64

(19922, 6)

Mes/Año
03/2025    4005
04/2025    3991
05/2025    3931
06/2025    3992
07/2025    4003
Name: count, dtype: int64

Estimo la regresión con la misma especificación que antes: mes, edad, sexo y estudios, usando `Peso` y errores robustos.


In [11]:
formula_reg2 = construir_formula_base("practica_religion_si_no")
modelo_reg2 = estimar_wls_base(df_reg2, "practica_religion_si_no")


Reviso la media ponderada por mes y el resumen del modelo para comparar la evolución bruta con la estimación controlada.


In [12]:
display(tabla_resumen_dependiente(df_reg2, "practica_religion_si_no"))
modelo_reg2.summary()


,Mes/Año,n,peso_total,media_ponderada
0,03/2025,4005,3995.4616,0.2967
1,04/2025,3991,3971.8615,0.3079
2,05/2025,3931,3883.8515,0.3218
3,06/2025,3992,3981.8139,0.2850
4,07/2025,4003,3979.7615,0.2935


<class 'statsmodels.iolib.summary.Summary'>
"""
                               WLS Regression Results                              
===================================================================================
Dep. Variable:     practica_religion_si_no   R-squared:                       0.048
Model:                                 WLS   Adj. R-squared:                  0.047
Method:                      Least Squares   F-statistic:                     30.34
Date:                     Thu, 21 May 2026   Prob (F-statistic):           7.04e-70
Time:                             14:07:01   Log-Likelihood:                -15474.
No. Observations:                    19922   AIC:                         3.097e+04
Df Residuals:                        19909   BIC:                         3.108e+04
Df Model:                               12                                         
Covariance Type:                       HC1                                         
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              0.0231      0.018      1.253      0.210      -0.013       0.059
C(Q("Mes/Año"))[T.04/2025]             0.0112      0.014      0.776      0.438      -0.017       0.040
C(Q("Mes/Año"))[T.05/2025]             0.0271      0.014      1.883      0.060      -0.001       0.055
C(Q("Mes/Año"))[T.06/2025]            -0.0115      0.014     -0.801      0.423      -0.040       0.017
C(Q("Mes/Año"))[T.07/2025]            -0.0023      0.015     -0.154      0.878      -0.032       0.027
C(Sexo)[T.Mujer]                       0.0797      0.009      8.613      0.000       0.062       0.098
C(Estudios)[T.Otros]                   0.3596      0.233      1.545      0.122      -0.097       0.816
C(Estudios)[T.Primaria]                0.0991      0.026      3.820      0.000       0.048       0.150
C(Estudios)[T.Secundaria 1ª etapa]     0.0412      0.014      2.975      0.003       0.014       0.068
C(Estudios)[T.Secundaria 2ª etapa]     0.0261      0.012      2.160      0.031       0.002       0.050
C(Estudios)[T.Sin estudios]            0.0834      0.039      2.145      0.032       0.007       0.160
C(Estudios)[T.Superiores]              0.0081      0.009      0.878      0.380      -0.010       0.026
Edad                                   0.0039      0.000     13.196      0.000       0.003       0.004
==============================================================================
Omnibus:                     2674.312   Durbin-Watson:                   1.996
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             3936.019
Skew:                           1.003   Prob(JB):                         0.00
Kurtosis:                       3.848   Cond. No.                     1.56e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
[2] The condition number is large, 1.56e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Conclusiones de la regresión 2:


La práctica religiosa también alcanza su coeficiente más alto en mayo, pero la señal es bastante más débil que en religiosidad declarada. El coeficiente de mayo es `+0,0271`, mientras que abril, junio y julio no muestran cambios claros frente a marzo.


La lectura es prudente. La religiosidad declarada sí muestra un salto claro en mayo, mientras que la práctica religiosa apenas ofrece una señal más ajustada. Esto tiene sentido teórico: una identidad puede activarse con más facilidad que un hábito de comportamiento.

Por tanto, hasta aquí el shock parece reflejarse sobre todo en la dimensión identitaria de la religión, y solo de forma más limitada en la práctica.


## Regresiones 3-6: intención de voto

Una vez comprobado si el shock aparece en la dimensión religiosa, paso a las variables políticas. Aquí no busco demostrar que todo cambio electoral venga provocado por la religión. La pregunta es más acotada: si en los meses posteriores al shock aparecen movimientos en intención de voto que puedan leerse junto con la activación religiosa observada antes.

Las cuatro variables binarias de voto (`voto_psoe_si_no`, `voto_pp_si_no`, `voto_vox_si_no` y `voto_sumar_si_no`) ya vienen construidas en la base preparada. En cada caso, el valor 1 indica intención de votar a ese partido y el valor 0 cualquier otra respuesta válida de voto.


Estimo las cuatro regresiones con la misma especificación que en los modelos religiosos. Marzo vuelve a ser el mes base, de modo que cada coeficiente mensual indica la diferencia respecto al periodo previo al shock.


In [13]:
def estimar_modelo_voto(variable_dependiente):
    variables = [
        variable_dependiente,
        "Mes/Año",
        "Edad",
        "Sexo",
        "Estudios",
        "Peso",
    ]

    df_reg = df_total[variables].dropna().copy()
    df_reg[variable_dependiente] = df_reg[variable_dependiente].astype(int)

    formula = construir_formula_base(variable_dependiente)
    modelo = estimar_wls_base(df_reg, variable_dependiente)

    return df_reg, modelo


df_reg3, modelo_reg3 = estimar_modelo_voto("voto_psoe_si_no")
df_reg4, modelo_reg4 = estimar_modelo_voto("voto_pp_si_no")
df_reg5, modelo_reg5 = estimar_modelo_voto("voto_vox_si_no")
df_reg6, modelo_reg6 = estimar_modelo_voto("voto_sumar_si_no")

resumen_regresiones_voto = pd.DataFrame({
    "regresion": ["reg3", "reg4", "reg5", "reg6"],
    "variable_dependiente": ["voto_psoe_si_no", "voto_pp_si_no", "voto_vox_si_no", "voto_sumar_si_no"],
    "observaciones": [len(df_reg3), len(df_reg4), len(df_reg5), len(df_reg6)],
    "media_ponderada": [
        media_ponderada(df_reg3["voto_psoe_si_no"], df_reg3["Peso"]),
        media_ponderada(df_reg4["voto_pp_si_no"], df_reg4["Peso"]),
        media_ponderada(df_reg5["voto_vox_si_no"], df_reg5["Peso"]),
        media_ponderada(df_reg6["voto_sumar_si_no"], df_reg6["Peso"]),
    ],
}).round(4)

resumen_regresiones_voto


,regresion,variable_dependiente,observaciones,media_ponderada
0,reg3,voto_psoe_si_no,19450,0.2315
1,reg4,voto_pp_si_no,19450,0.1967
2,reg5,voto_vox_si_no,19450,0.1100
3,reg6,voto_sumar_si_no,19450,0.0486


Empiezo por el PSOE, para ver si su intención de voto cambia respecto a marzo una vez introducidos los controles.


In [14]:
modelo_reg3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:        voto_psoe_si_no   R-squared:                       0.028
Model:                            WLS   Adj. R-squared:                  0.027
Method:                 Least Squares   F-statistic:                     28.12
Date:                Thu, 21 May 2026   Prob (F-statistic):           2.51e-64
Time:                        14:07:02   Log-Likelihood:                -13674.
No. Observations:               19450   AIC:                         2.737e+04
Df Residuals:                   19437   BIC:                         2.748e+04
Df Model:                          12                                         
Covariance Type:                  HC1                                         
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              0.0395      0.017      2.370      0.018       0.007       0.072
C(Q("Mes/Año"))[T.04/2025]             0.0034      0.014      0.248      0.804      -0.024       0.031
C(Q("Mes/Año"))[T.05/2025]            -0.0078      0.013     -0.586      0.558      -0.034       0.018
C(Q("Mes/Año"))[T.06/2025]             0.0046      0.014      0.328      0.743      -0.023       0.032
C(Q("Mes/Año"))[T.07/2025]            -0.0541      0.013     -4.082      0.000      -0.080      -0.028
C(Sexo)[T.Mujer]                       0.0332      0.009      3.846      0.000       0.016       0.050
C(Estudios)[T.Otros]                  -0.2342      0.027     -8.800      0.000      -0.286      -0.182
C(Estudios)[T.Primaria]               -0.0077      0.024     -0.329      0.742      -0.054       0.038
C(Estudios)[T.Secundaria 1ª etapa]     0.0087      0.012      0.697      0.486      -0.016       0.033
C(Estudios)[T.Secundaria 2ª etapa]     0.0237      0.011      2.111      0.035       0.002       0.046
C(Estudios)[T.Sin estudios]            0.0362      0.038      0.954      0.340      -0.038       0.110
C(Estudios)[T.Superiores]              0.0061      0.008      0.721      0.471      -0.011       0.023
Edad                                   0.0035      0.000     13.150      0.000       0.003       0.004
==============================================================================
Omnibus:                     5320.665   Durbin-Watson:                   2.003
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            12181.525
Skew:                           1.565   Prob(JB):                         0.00
Kurtosis:                       5.287   Cond. No.                     1.54e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
[2] The condition number is large, 1.54e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Conclusiones regresión 3: PSOE


En el PSOE no aparece ningún cambio claro entre abril y junio. El movimiento relevante llega en julio, con un coeficiente de `-0,0541`.


Esta caída de julio debe interpretarse con cautela. Es un resultado políticamente importante, pero aparece bastante después del momento de máxima saliencia religiosa. Por eso no conviene leerlo como una consecuencia directa del shock religioso sin tener en cuenta otros acontecimientos del periodo.


Ahora reviso el modelo equivalente para el PP.


In [15]:
modelo_reg4.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:          voto_pp_si_no   R-squared:                       0.012
Model:                            WLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     13.70
Date:                Thu, 21 May 2026   Prob (F-statistic):           9.09e-29
Time:                        14:07:02   Log-Likelihood:                -12677.
No. Observations:               19450   AIC:                         2.538e+04
Df Residuals:                   19437   BIC:                         2.548e+04
Df Model:                          12                                         
Covariance Type:                  HC1                                         
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              0.0928      0.015      6.047      0.000       0.063       0.123
C(Q("Mes/Año"))[T.04/2025]            -0.0038      0.012     -0.314      0.753      -0.028       0.020
C(Q("Mes/Año"))[T.05/2025]             0.0141      0.012      1.157      0.247      -0.010       0.038
C(Q("Mes/Año"))[T.06/2025]            -0.0077      0.012     -0.634      0.526      -0.031       0.016
C(Q("Mes/Año"))[T.07/2025]            -0.0020      0.013     -0.160      0.873      -0.027       0.023
C(Sexo)[T.Mujer]                      -0.0254      0.008     -3.240      0.001      -0.041      -0.010
C(Estudios)[T.Otros]                   0.0621      0.229      0.272      0.786      -0.386       0.510
C(Estudios)[T.Primaria]               -0.0254      0.021     -1.214      0.225      -0.066       0.016
C(Estudios)[T.Secundaria 1ª etapa]    -0.0056      0.012     -0.464      0.643      -0.029       0.018
C(Estudios)[T.Secundaria 2ª etapa]     0.0244      0.011      2.249      0.025       0.003       0.046
C(Estudios)[T.Sin estudios]           -0.1030      0.026     -3.897      0.000      -0.155      -0.051
C(Estudios)[T.Superiores]              0.0590      0.009      6.933      0.000       0.042       0.076
Edad                                   0.0020      0.000      8.499      0.000       0.002       0.003
==============================================================================
Omnibus:                     7267.600   Durbin-Watson:                   1.988
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            23830.830
Skew:                           1.951   Prob(JB):                         0.00
Kurtosis:                       6.767   Cond. No.                     1.54e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
[2] The condition number is large, 1.54e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Conclusiones regresión 4: PP


En el PP no aparece ningún cambio claro en ninguno de los meses. Los coeficientes son pequeños y no muestran una diferencia nítida frente a marzo.


Esto es relevante porque el PP tiene un electorado más religioso, pero eso no se traduce aquí en un aumento agregado claro de intención de voto tras el shock. De nuevo, conviene separar diferencias de nivel entre electorados y cambios temporales atribuibles al periodo.


Paso ahora a VOX, donde interesa ver si aparecen movimientos en un partido cuyo electorado también presenta altos niveles de religiosidad.


In [16]:
modelo_reg5.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:         voto_vox_si_no   R-squared:                       0.055
Model:                            WLS   Adj. R-squared:                  0.054
Method:                 Least Squares   F-statistic:                     28.26
Date:                Thu, 21 May 2026   Prob (F-statistic):           1.10e-64
Time:                        14:07:02   Log-Likelihood:                -7596.6
No. Observations:               19450   AIC:                         1.522e+04
Df Residuals:                   19437   BIC:                         1.532e+04
Df Model:                          12                                         
Covariance Type:                  HC1                                         
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              0.2918      0.016     18.356      0.000       0.261       0.323
C(Q("Mes/Año"))[T.04/2025]             0.0247      0.011      2.314      0.021       0.004       0.046
C(Q("Mes/Año"))[T.05/2025]             0.0118      0.010      1.202      0.229      -0.007       0.031
C(Q("Mes/Año"))[T.06/2025]             0.0036      0.010      0.370      0.712      -0.015       0.022
C(Q("Mes/Año"))[T.07/2025]             0.0528      0.012      4.505      0.000       0.030       0.076
C(Sexo)[T.Mujer]                      -0.0509      0.007     -7.380      0.000      -0.064      -0.037
C(Estudios)[T.Otros]                   0.0679      0.163      0.416      0.678      -0.252       0.388
C(Estudios)[T.Primaria]                0.0649      0.020      3.280      0.001       0.026       0.104
C(Estudios)[T.Secundaria 1ª etapa]     0.0445      0.012      3.806      0.000       0.022       0.067
C(Estudios)[T.Secundaria 2ª etapa]     0.0038      0.010      0.398      0.691      -0.015       0.023
C(Estudios)[T.Sin estudios]            0.0568      0.021      2.725      0.006       0.016       0.098
C(Estudios)[T.Superiores]             -0.0538      0.007     -7.967      0.000      -0.067      -0.041
Edad                                  -0.0035      0.000    -13.196      0.000      -0.004      -0.003
==============================================================================
Omnibus:                    13233.418   Durbin-Watson:                   2.012
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           164530.987
Skew:                           3.225   Prob(JB):                         0.00
Kurtosis:                      15.705   Cond. No.                     1.54e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
[2] The condition number is large, 1.54e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Conclusiones regresión 5: VOX


En VOX sí aparecen dos aumentos claros: abril (`+0,0247`) y julio (`+0,0528`). Mayo y junio, en cambio, no muestran cambios estadísticamente claros frente a marzo.


La señal de VOX existe, pero no sigue una trayectoria sencilla de aumento justo después del shock. El repunte de abril es anterior al momento de máxima saliencia de mayo, y el de julio aparece en un contexto político más cargado. Por tanto, este resultado es interesante para la discusión, pero no basta por sí solo para afirmar una traslación directa del shock religioso al voto.


Por último, reviso el modelo de Sumar.


In [17]:
modelo_reg6.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:       voto_sumar_si_no   R-squared:                       0.010
Model:                            WLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     12.78
Date:                Thu, 21 May 2026   Prob (F-statistic):           1.50e-26
Time:                        14:07:02   Log-Likelihood:                -748.34
No. Observations:               19450   AIC:                             1523.
Df Residuals:                   19437   BIC:                             1625.
Df Model:                          12                                         
Covariance Type:                  HC1                                         
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              0.0700      0.007      9.786      0.000       0.056       0.084
C(Q("Mes/Año"))[T.04/2025]            -0.0067      0.006     -1.184      0.237      -0.018       0.004
C(Q("Mes/Año"))[T.05/2025]            -0.0053      0.006     -0.879      0.379      -0.017       0.006
C(Q("Mes/Año"))[T.06/2025]            -0.0043      0.006     -0.739      0.460      -0.016       0.007
C(Q("Mes/Año"))[T.07/2025]             0.0040      0.007      0.591      0.554      -0.009       0.017
C(Sexo)[T.Mujer]                      -0.0015      0.004     -0.408      0.683      -0.009       0.006
C(Estudios)[T.Otros]                   0.1655      0.185      0.893      0.372      -0.198       0.529
C(Estudios)[T.Primaria]               -0.0197      0.008     -2.568      0.010      -0.035      -0.005
C(Estudios)[T.Secundaria 1ª etapa]    -0.0109      0.006     -1.847      0.065      -0.022       0.001
C(Estudios)[T.Secundaria 2ª etapa]     0.0032      0.006      0.532      0.595      -0.009       0.015
C(Estudios)[T.Sin estudios]           -0.0117      0.014     -0.850      0.395      -0.039       0.015
C(Estudios)[T.Superiores]              0.0269      0.005      5.654      0.000       0.018       0.036
Edad                                  -0.0004      0.000     -3.639      0.000      -0.001      -0.000
==============================================================================
Omnibus:                    18440.414   Durbin-Watson:                   1.975
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           632039.518
Skew:                           4.781   Prob(JB):                         0.00
Kurtosis:                      29.238   Cond. No.                     1.54e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
[2] The condition number is large, 1.54e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Conclusiones regresión 6: Sumar


En Sumar no aparece ningún cambio claro. Los coeficientes son pequeños y no muestran diferencias relevantes frente a marzo.


La intención de voto a Sumar se mantiene bastante estable en este bloque de regresiones. Esto ayuda a cerrar la primera lectura política: no todos los partidos muestran movimientos, y los cambios observados no forman un patrón simple que pueda atribuirse automáticamente al shock religioso.


## Regresión 7: ideología

Incluyo también la ideología como variable dependiente. Esta regresión no es el centro del trabajo, pero ayuda a contextualizar los resultados de voto. Si durante el periodo hubiera un desplazamiento ideológico general, podría confundirse con cambios atribuibles a la religión. Por eso conviene comprobar si la escala izquierda-derecha se mueve de forma clara respecto a marzo.


In [18]:
variables_reg7 = [
    "Ideologia",
    "Mes/Año",
    "Edad",
    "Sexo",
    "Estudios",
    "Peso",
]

df_reg7 = df_total[variables_reg7].dropna().copy()
df_reg7["Ideologia"] = df_reg7["Ideologia"].astype(int)

display(df_reg7.isna().sum())
display(df_reg7.shape)
display(df_reg7["Mes/Año"].value_counts().sort_index())

Ideologia    0
Mes/Año      0
Edad         0
Sexo         0
Estudios     0
Peso         0
dtype: int64

(19317, 6)

Mes/Año
03/2025    3847
04/2025    3871
05/2025    3856
06/2025    3845
07/2025    3898
Name: count, dtype: int64

Estimo la regresión de ideología con la misma especificación que el resto y muestro la media ponderada por mes para facilitar la lectura.


In [19]:
formula_reg7 = construir_formula_base("Ideologia")
modelo_reg7 = estimar_wls_base(df_reg7, "Ideologia")

display(tabla_resumen_dependiente(df_reg7, "Ideologia"))
modelo_reg7.summary()


,Mes/Año,n,peso_total,media_ponderada
0,03/2025,3847,3764.1605,5.0201
1,04/2025,3871,3778.8640,4.7774
2,05/2025,3856,3771.4279,4.9283
3,06/2025,3845,3769.6234,4.7305
4,07/2025,3898,3791.0138,4.9577


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:              Ideologia   R-squared:                       0.012
Model:                            WLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     10.02
Date:                Thu, 21 May 2026   Prob (F-statistic):           6.64e-20
Time:                        14:07:02   Log-Likelihood:                -49008.
No. Observations:               19317   AIC:                         9.804e+04
Df Residuals:                   19304   BIC:                         9.814e+04
Df Model:                          12                                         
Covariance Type:                  HC1                                         
======================================================================================================
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                              5.3777      0.120     44.959      0.000       5.143       5.612
C(Q("Mes/Año"))[T.04/2025]            -0.2439      0.091     -2.687      0.007      -0.422      -0.066
C(Q("Mes/Año"))[T.05/2025]            -0.0852      0.090     -0.950      0.342      -0.261       0.090
C(Q("Mes/Año"))[T.06/2025]            -0.2837      0.093     -3.043      0.002      -0.466      -0.101
C(Q("Mes/Año"))[T.07/2025]            -0.0529      0.097     -0.546      0.585      -0.243       0.137
C(Sexo)[T.Mujer]                      -0.1539      0.059     -2.627      0.009      -0.269      -0.039
C(Estudios)[T.Otros]                   2.3063      0.956      2.413      0.016       0.433       4.180
C(Estudios)[T.Primaria]                0.4341      0.178      2.439      0.015       0.085       0.783
C(Estudios)[T.Secundaria 1ª etapa]     0.3050      0.089      3.436      0.001       0.131       0.479
C(Estudios)[T.Secundaria 2ª etapa]     0.0573      0.072      0.797      0.425      -0.084       0.198
C(Estudios)[T.Sin estudios]           -0.0379      0.289     -0.131      0.896      -0.605       0.529
C(Estudios)[T.Superiores]             -0.2333      0.055     -4.256      0.000      -0.341      -0.126
Edad                                  -0.0069      0.002     -3.541      0.000      -0.011      -0.003
==============================================================================
Omnibus:                     1856.750   Durbin-Watson:                   1.970
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             8341.138
Skew:                           0.381   Prob(JB):                         0.00
Kurtosis:                       6.128   Cond. No.                     1.36e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
[2] The condition number is large, 1.36e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Conclusiones regresión 7: ideología


La regresión de ideología no muestra un desplazamiento claro hacia posiciones más a la derecha tras el shock. Como la escala va de 1 izquierda a 10 derecha, los coeficientes negativos indican una posición media algo más a la izquierda que en marzo. Abril presenta un coeficiente de `-0,2439` y junio de `-0,2837`; mayo y julio no muestran cambios claros.

Esto refuerza una idea importante: no parece que los resultados religiosos puedan explicarse simplemente por un giro ideológico agregado hacia la derecha. La ideología se mueve de forma irregular y probablemente recoge otros elementos del contexto político y social.


## Resumen conjunto de regresiones base

Antes de pasar a análisis más específicos, reúno en una sola tabla los coeficientes mensuales de las regresiones base. Esta tabla sirve como mapa del análisis: muestra dónde aparecen señales claras y dónde no, siempre comparando cada mes con marzo de 2025.


In [20]:
modelos_base = [
    ("Regresión 1", "Religiosidad declarada", "religiosidad_si_no", df_reg1, modelo_reg1),
    ("Regresión 2", "Práctica religiosa", "practica_religion_si_no", df_reg2, modelo_reg2),
    ("Regresión 3", "Voto PSOE", "voto_psoe_si_no", df_reg3, modelo_reg3),
    ("Regresión 4", "Voto PP", "voto_pp_si_no", df_reg4, modelo_reg4),
    ("Regresión 5", "Voto VOX", "voto_vox_si_no", df_reg5, modelo_reg5),
    ("Regresión 6", "Voto Sumar", "voto_sumar_si_no", df_reg6, modelo_reg6),
    ("Regresión 7", "Ideología", "Ideologia", df_reg7, modelo_reg7),
]

resumen_modelos_base = pd.DataFrame([
    {
        "regresion": regresion,
        "variable": variable,
        "variable_dependiente": variable_dependiente,
        "observaciones": len(df_modelo),
        "media_ponderada": media_ponderada(df_modelo[variable_dependiente], df_modelo["Peso"]),
        "r2": modelo.rsquared,
    }
    for regresion, variable, variable_dependiente, df_modelo, modelo in modelos_base
]).round(4)

coeficientes_meses_base = pd.concat(
    [
        extraer_coeficientes_meses(modelo, regresion, variable)
        for regresion, variable, variable_dependiente, df_modelo, modelo in modelos_base
    ],
    ignore_index=True,
)

coeficientes_meses_base["coeficiente_formateado"] = coeficientes_meses_base.apply(
    lambda fila: formatear_coeficiente(fila["coeficiente"], fila["p_value"]),
    axis=1,
)

tabla_coeficientes_base = (
    coeficientes_meses_base
    .pivot_table(
        index=["regresion", "variable"],
        columns="mes",
        values="coeficiente_formateado",
        aggfunc="first",
    )
    .reset_index()
)

display(resumen_modelos_base)
tabla_coeficientes_base


,regresion,variable,variable_dependiente,observaciones,media_ponderada,r2
0,Regresión 1,Religiosidad declarada,religiosidad_si_no,20057,0.5615,0.0714
1,Regresión 2,Práctica religiosa,practica_religion_si_no,19922,0.3009,0.0477
2,Regresión 3,Voto PSOE,voto_psoe_si_no,19450,0.2315,0.0280
3,Regresión 4,Voto PP,voto_pp_si_no,19450,0.1967,0.0122
4,Regresión 5,Voto VOX,voto_vox_si_no,19450,0.1100,0.0546
5,Regresión 6,Voto Sumar,voto_sumar_si_no,19450,0.0486,0.0096
6,Regresión 7,Ideología,Ideologia,19317,4.8828,0.0122


mes,regresion,variable,04/2025,05/2025,06/2025,07/2025
0,Regresión 1,Religiosidad declarada,0.0116,0.0410***,0.0168,0.0187
1,Regresión 2,Práctica religiosa,0.0112,0.0271*,-0.0115,-0.0023
2,Regresión 3,Voto PSOE,0.0034,-0.0078,0.0046,-0.0541***
3,Regresión 4,Voto PP,-0.0038,0.0141,-0.0077,-0.0020
4,Regresión 5,Voto VOX,0.0247**,0.0118,0.0036,0.0528***
5,Regresión 6,Voto Sumar,-0.0067,-0.0053,-0.0043,0.0040
6,Regresión 7,Ideología,-0.2439***,-0.0852,-0.2837***,-0.0529


Los asteriscos indican niveles convencionales de significatividad: `*` p < 0,10, `**` p < 0,05 y `***` p < 0,01. Esta tabla no sustituye la interpretación de cada modelo, pero ayuda a ordenar los resultados antes de seguir avanzando.


# 2. Composición religiosa de los electorados

Después de las regresiones base, incorporo un análisis complementario sobre la composición religiosa de los electorados. Este punto debe entenderse bien: no busco demostrar que el PP o VOX son más religiosos que PSOE o Sumar, porque eso ya se ha visto en el análisis descriptivo y es una diferencia de nivel bastante clara. Lo que interesa ahora es si esa composición cambia durante los meses del shock.

La lógica es la siguiente: si el shock activa la identidad religiosa, podría cambiar la proporción de personas religiosas dentro de algunos electorados declarados. Esto no prueba que los mismos individuos cambien de voto, porque los barómetros no son paneles. Pero sí puede mostrar si, en la ventana posterior al shock, algunos espacios electorales aparecen más o menos asociados a la religiosidad que en marzo.


Empiezo con una tabla descriptiva ponderada. Para cada partido y mes, calculo qué porcentaje de su electorado declarado se identifica como religioso. También muestro el tamaño muestral de cada celda, porque algunos electorados tienen menos observaciones y eso obliga a ser más prudente.


In [21]:
partidos_interes = ["PSOE", "PP", "VOX", "Sumar"]

df_electorados = df_total[df_total["Intencion_voto"].isin(partidos_interes)].copy()

tabla_n_electorados = (
    df_electorados
    .pivot_table(
        index="Intencion_voto",
        columns="Mes/Año",
        values="religiosidad_si_no",
        aggfunc="size",
    )
    .loc[partidos_interes]
)

tabla_religiosidad_partido_mes_pond = (
    df_electorados
    .groupby(["Intencion_voto", "Mes/Año"])
    .apply(lambda grupo: media_ponderada(grupo["religiosidad_si_no"], grupo["Peso"]) * 100)
    .unstack()
    .loc[partidos_interes]
    .round(2)
)

display(tabla_n_electorados)
tabla_religiosidad_partido_mes_pond


Mes/Año,03/2025,04/2025,05/2025,06/2025,07/2025
Intencion_voto,,,,,
PSOE,937,916,927,905,755
PP,879,827,893,801,876
VOX,298,335,337,343,425
Sumar,235,224,203,224,245


Mes/Año,03/2025,04/2025,05/2025,06/2025,07/2025
Intencion_voto,,,,,
PSOE,47.48,45.69,51.03,50.27,47.51
PP,78.47,78.22,84.23,77.22,79.53
VOX,64.69,71.51,68.04,67.28,68.23
Sumar,17.59,17.40,19.91,21.52,28.45


Después estimo una regresión separada para cada electorado. La variable dependiente vuelve a ser `religiosidad_si_no`, y la especificación mantiene los mismos controles que antes. Así puedo ver si, dentro de cada partido, la proporción de personas religiosas cambia respecto a marzo.


In [22]:
def estimar_religiosidad_en_electorado(partido):
    variables = [
        "religiosidad_si_no",
        "Mes/Año",
        "Edad",
        "Sexo",
        "Estudios",
        "Peso",
    ]

    df_partido = (
        df_electorados
        .loc[df_electorados["Intencion_voto"].eq(partido), variables]
        .dropna()
        .copy()
    )

    modelo = estimar_wls_base(df_partido, "religiosidad_si_no")
    return df_partido, modelo


modelos_religiosidad_electorados = {}

for partido in partidos_interes:
    df_partido, modelo_partido = estimar_religiosidad_en_electorado(partido)
    modelos_religiosidad_electorados[partido] = {
        "df": df_partido,
        "modelo": modelo_partido,
    }

resumen_religiosidad_electorados = pd.DataFrame([
    {
        "partido": partido,
        "observaciones": len(resultado["df"]),
        "media_ponderada": media_ponderada(resultado["df"]["religiosidad_si_no"], resultado["df"]["Peso"]),
        "r2": resultado["modelo"].rsquared,
    }
    for partido, resultado in modelos_religiosidad_electorados.items()
]).round(4)

coeficientes_religiosidad_electorados = []

for partido, resultado in modelos_religiosidad_electorados.items():
    coeficientes_partido = extraer_coeficientes_meses(
        resultado["modelo"],
        partido,
        "religiosidad_si_no",
    ).rename(columns={"regresion": "partido"})
    coeficientes_religiosidad_electorados.append(coeficientes_partido)

coeficientes_religiosidad_electorados = pd.concat(
    coeficientes_religiosidad_electorados,
    ignore_index=True,
)

coeficientes_religiosidad_electorados["coeficiente_formateado"] = coeficientes_religiosidad_electorados.apply(
    lambda fila: formatear_coeficiente(fila["coeficiente"], fila["p_value"]),
    axis=1,
)

tabla_regresiones_religiosidad_electorados = (
    coeficientes_religiosidad_electorados
    .pivot_table(
        index="partido",
        columns="mes",
        values="coeficiente_formateado",
        aggfunc="first",
    )
    .loc[partidos_interes]
    .reset_index()
)

display(resumen_religiosidad_electorados)
tabla_regresiones_religiosidad_electorados


,partido,observaciones,media_ponderada,r2
0,PSOE,4437,0.4833,0.1164
1,PP,4276,0.7963,0.0626
2,VOX,1736,0.6804,0.0698
3,Sumar,1130,0.2128,0.0927


mes,partido,04/2025,05/2025,06/2025,07/2025
0,PSOE,-0.0204,0.0299,0.0098,-0.0108
1,PP,-0.0150,0.0543**,-0.0147,0.0054
2,VOX,0.0678,0.0268,0.0246,0.0336
3,Sumar,-0.0137,0.0129,0.0302,0.0900*


Por último, estimo un modelo conjunto con interacción entre mes e intención de voto. Esta parte no busca repetir lo obvio de las diferencias entre partidos, sino comprobar si los cambios temporales de religiosidad son distintos entre electorados. En este modelo, PSOE actúa como categoría de referencia.


In [23]:
variables_interaccion_electorados = [
    "religiosidad_si_no",
    "Mes/Año",
    "Intencion_voto",
    "Edad",
    "Sexo",
    "Estudios",
    "Peso",
]

df_interaccion_electorados = df_electorados[variables_interaccion_electorados].dropna().copy()

formula_interaccion_electorados = (
    'religiosidad_si_no ~ '
    'C(Q("Mes/Año"), Treatment(reference="03/2025")) '
    '* C(Intencion_voto, Treatment(reference="PSOE")) '
    '+ Edad + C(Sexo) + C(Estudios)'
)

modelo_interaccion_electorados = smf.wls(
    formula=formula_interaccion_electorados,
    data=df_interaccion_electorados,
    weights=df_interaccion_electorados["Peso"],
).fit(cov_type="HC1")

filas_interaccion = []

for partido in ["PP", "VOX", "Sumar"]:
    for mes in MESES_COMPARACION:
        termino = (
            f'C(Q("Mes/Año"), Treatment(reference="03/2025"))[T.{mes}]:'
            f'C(Intencion_voto, Treatment(reference="PSOE"))[T.{partido}]'
        )
        if termino in modelo_interaccion_electorados.params.index:
            filas_interaccion.append({
                "partido": partido,
                "mes": mes,
                "coeficiente": modelo_interaccion_electorados.params[termino],
                "p_value": modelo_interaccion_electorados.pvalues[termino],
            })

tabla_interacciones_religiosidad_electorados = pd.DataFrame(filas_interaccion)

tabla_interacciones_religiosidad_electorados["coeficiente_formateado"] = tabla_interacciones_religiosidad_electorados.apply(
    lambda fila: formatear_coeficiente(fila["coeficiente"], fila["p_value"]),
    axis=1,
)

tabla_interacciones_religiosidad_electorados = (
    tabla_interacciones_religiosidad_electorados
    .pivot_table(
        index="partido",
        columns="mes",
        values="coeficiente_formateado",
        aggfunc="first",
    )
    .reset_index()
)

display(tabla_interacciones_religiosidad_electorados)
modelo_interaccion_electorados.summary()


mes,partido,04/2025,05/2025,06/2025,07/2025
0,PP,-0.0038,0.0245,-0.0313,0.0167
1,Sumar,0.0286,0.0015,0.0376,0.1305**
2,VOX,0.0781,0.0047,0.0238,0.0404


<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:     religiosidad_si_no   R-squared:                       0.196
Model:                            WLS   Adj. R-squared:                  0.194
Method:                 Least Squares   F-statistic:                     79.38
Date:                Thu, 21 May 2026   Prob (F-statistic):               0.00
Time:                        14:07:02   Log-Likelihood:                -8785.3
No. Observations:               11579   AIC:                         1.763e+04
Df Residuals:                   11551   BIC:                         1.783e+04
Df Model:                          27                                         
Covariance Type:                  HC1                                         
======================================================================================================================================================================================
                                                                                                                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                                                              0.1368      0.031      4.361      0.000       0.075       0.198
C(Q("Mes/Año"), Treatment(reference="03/2025"))[T.04/2025]                                                            -0.0172      0.031     -0.547      0.584      -0.079       0.044
C(Q("Mes/Año"), Treatment(reference="03/2025"))[T.05/2025]                                                             0.0299      0.031      0.977      0.329      -0.030       0.090
C(Q("Mes/Año"), Treatment(reference="03/2025"))[T.06/2025]                                                             0.0168      0.032      0.524      0.600      -0.046       0.080
C(Q("Mes/Año"), Treatment(reference="03/2025"))[T.07/2025]                                                            -0.0108      0.033     -0.326      0.744      -0.076       0.054
C(Intencion_voto, Treatment(reference="PSOE"))[T.PP]                                                                   0.3383      0.028     12.006      0.000       0.283       0.394
C(Intencion_voto, Treatment(reference="PSOE"))[T.Sumar]                                                               -0.2416      0.044     -5.546      0.000      -0.327      -0.156
C(Intencion_voto, Treatment(reference="PSOE"))[T.VOX]                                                                  0.2484      0.045      5.576      0.000       0.161       0.336
C(Sexo)[T.Mujer]                                                                                                       0.0641      0.012      5.258      0.000       0.040       0.088
C(Estudios)[T.Otros]                                                                                                   0.4735      0.139      3.399      0.001       0.200       0.746
C(Estudios)[T.Primaria]                                                                                                0.0535      0.032      1.699      0.089      -0.008       0.115
C(Estudios)[T.Secundaria 1ª etapa]                                                                                     0.0330      0.019      1.722      0.085      -0.005       0.070
C(Estudios)[T.Secundaria 2ª etapa]                                                                                    -0.0410      0.017     -2.480      0.013      -0.073      -0.009
C(Estudios)[T.Sin estudios]                                                                                            0.0422      0.045      0.934      0.

Lo primero que se observa es la diferencia de nivel ya esperada: los electorados de PP y VOX son más religiosos que los de PSOE y Sumar. Pero lo más relevante para este análisis no es esa diferencia estática, sino el movimiento temporal dentro de cada partido.

En el PP aparece un aumento claro en mayo: la proporción ponderada de personas religiosas pasa del `78,47%` en marzo al `84,23%` en mayo, y la regresión separada estima un coeficiente de `+0,0543`. En Sumar el cambio más visible aparece más tarde: pasa del `17,59%` en marzo al `28,45%` en julio, con un coeficiente de `+0,0900`, aunque aquí el tamaño muestral es menor y la lectura debe ser más prudente. En PSOE y VOX no aparecen cambios claros frente a marzo.

El modelo con interacción confirma que la diferencia más llamativa aparece en Sumar en julio. Esta evidencia complementa las regresiones base: la señal religiosa es clara en mayo a nivel agregado, pero su posible traducción política aparece de forma parcial, desigual y no inmediata. También, analizaremos una posible hipótesis de que este resultado puede deberse a que en los colectivos más de izquierdas, el shock supone un cambio mayor ya que normalmente tienen menos contacto con el tema religioso y por eso les afecta más.

# 3. Percepción de problemas: hipótesis a partir de la religiosidad

En este bloque vuelvo al segundo resultado que interesa al TFG: la percepción de los principales problemas. Para no hacer un análisis demasiado disperso, no voy a estudiar todas las categorías posibles, sino solo aquellas que en el descriptivo aparecían más claramente diferenciadas por religiosidad.

El punto de partida es el siguiente: en el análisis exploratorio vimos que la vivienda pesa más entre las personas no religiosas, mientras que la inmigración, el paro y los problemas vinculados al Gobierno, partidos y políticos pesan más entre las personas religiosas. A partir de ahí, la hipótesis es sencilla: si el shock religioso aumenta la saliencia de la dimensión religiosa, podríamos esperar una caída relativa de la vivienda y un aumento de esos problemas más presentes entre los religiosos.

Esto no significa que esos problemas sean religiosos en sí mismos. La idea es más limitada: comprobar si, después del shock, la agenda de problemas se mueve en la dirección que sugerían las diferencias iniciales entre religiosos y no religiosos. Por tanto, la pregunta de este bloque es concreta: respecto a marzo, ¿baja la mención de vivienda y suben inmigración, paro y Gobierno/partidos?


Primero construyo cuatro variables binarias. Cada una toma valor 1 si la persona menciona ese problema en `Problema_1`, `Problema_2` o `Problema_3`, y 0 si no lo menciona en ninguna de las tres respuestas.


In [24]:
columnas_problemas = ["Problema_1", "Problema_2", "Problema_3"]

problemas_hipotesis = {
    "problema_vivienda": {
        "problema": "La vivienda",
        "grupo_mayor": "No religiosos",
        "direccion_esperada": "Baja tras el shock",
    },
    "problema_inmigracion": {
        "problema": "La inmigración",
        "grupo_mayor": "Religiosos",
        "direccion_esperada": "Sube tras el shock",
    },
    "problema_paro": {
        "problema": "El paro",
        "grupo_mayor": "Religiosos",
        "direccion_esperada": "Sube tras el shock",
    },
    "problema_gobierno_partidos": {
        "problema": "El Gobierno y partidos o políticos/as concretos/as",
        "grupo_mayor": "Religiosos",
        "direccion_esperada": "Sube tras el shock",
    },
}

for variable, info in problemas_hipotesis.items():
    df_total[variable] = df_total[columnas_problemas].eq(info["problema"]).any(axis=1).astype(int)

resumen_problemas_hipotesis = pd.DataFrame([
    {
        "variable": variable,
        "problema": info["problema"],
        "grupo_mayor_esperado": info["grupo_mayor"],
        "direccion_esperada": info["direccion_esperada"],
        "porcentaje_ponderado_total": media_ponderada(df_total[variable], df_total["Peso"]) * 100,
    }
    for variable, info in problemas_hipotesis.items()
]).round(2)

resumen_problemas_hipotesis


,variable,problema,grupo_mayor_esperado,direccion_esperada,porcentaje_ponderado_total
0,problema_vivienda,La vivienda,No religiosos,Baja tras el shock,29.02
1,problema_inmigracion,La inmigración,Religiosos,Sube tras el shock,17.67
2,problema_paro,El paro,Religiosos,Sube tras el shock,17.68
3,problema_gobierno_partidos,El Gobierno y partidos o políticos/as concreto...,Religiosos,Sube tras el shock,15.97


Antes de mirar la evolución temporal, compruebo que estos problemas efectivamente se distribuyen de forma distinta entre religiosos y no religiosos. Esta tabla reproduce la lógica del gráfico descriptivo: compara el porcentaje ponderado de cada problema en ambos grupos.


In [25]:
filas_diferencias_problemas = []

for variable, info in problemas_hipotesis.items():
    medias = {}
    for valor_grupo, etiqueta_grupo in [(0, "No religiosos"), (1, "Religiosos")]:
        datos_grupo = df_total[df_total["religiosidad_si_no"].eq(valor_grupo)]
        medias[etiqueta_grupo] = media_ponderada(datos_grupo[variable], datos_grupo["Peso"]) * 100

    filas_diferencias_problemas.append({
        "problema": info["problema"],
        "No religiosos": medias["No religiosos"],
        "Religiosos": medias["Religiosos"],
        "Diferencia religiosos - no religiosos": medias["Religiosos"] - medias["No religiosos"],
    })

tabla_diferencias_problemas_religiosidad = pd.DataFrame(filas_diferencias_problemas).round(2)

tabla_diferencias_problemas_religiosidad


,problema,No religiosos,Religiosos,Diferencia religiosos - no religiosos
0,La vivienda,35.40,24.04,-11.37
1,La inmigración,14.27,20.32,6.05
2,El paro,13.55,20.91,7.37
3,El Gobierno y partidos o políticos/as concreto...,9.36,21.13,11.77


Después miro la evolución mensual ponderada de estos cuatro problemas. Esto permite ver si, antes de controlar por edad, sexo y estudios, la dirección observada va en la línea esperada o no.


In [26]:
tabla_problemas_hipotesis_mes = pd.DataFrame({
    info["problema"]: (
        df_total
        .groupby("Mes/Año")
        .apply(lambda grupo: media_ponderada(grupo[variable], grupo["Peso"]) * 100)
    )
    for variable, info in problemas_hipotesis.items()
}).T.round(2)

tabla_problemas_hipotesis_mes


Mes/Año,03/2025,04/2025,05/2025,06/2025,07/2025
La vivienda,28.36,28.80,25.55,32.43,29.97
La inmigración,18.82,17.18,15.49,18.49,18.36
El paro,20.35,18.54,19.18,15.22,15.12
El Gobierno y partidos o políticos/as concretos/as,16.95,13.28,18.11,13.69,17.80


Por último, estimo cuatro regresiones, una para cada problema. En todas mantengo la misma especificación del análisis principal: mes, edad, sexo y estudios, con `Peso` y errores robustos.

La interpretación es directa: cada coeficiente mensual indica cuánto cambia la probabilidad de mencionar ese problema respecto a marzo, manteniendo constantes los controles básicos. Si la hipótesis se cumple, esperaríamos coeficientes negativos en vivienda y positivos en inmigración, paro y Gobierno/partidos en los meses posteriores al shock.


In [27]:
modelos_problemas_hipotesis = {}

for variable, info in problemas_hipotesis.items():
    variables_modelo = [
        variable,
        "Mes/Año",
        "Edad",
        "Sexo",
        "Estudios",
        "Peso",
    ]
    df_problema = df_total[variables_modelo].dropna().copy()
    modelo_problema = estimar_wls_base(df_problema, variable)
    modelos_problemas_hipotesis[variable] = {
        "problema": info["problema"],
        "direccion_esperada": info["direccion_esperada"],
        "df": df_problema,
        "modelo": modelo_problema,
    }

coeficientes_problemas_hipotesis = pd.concat(
    [
        extraer_coeficientes_meses(
            resultado["modelo"],
            resultado["problema"],
            variable,
        )
        for variable, resultado in modelos_problemas_hipotesis.items()
    ],
    ignore_index=True,
)

coeficientes_problemas_hipotesis["coeficiente_formateado"] = coeficientes_problemas_hipotesis.apply(
    lambda fila: formatear_coeficiente(fila["coeficiente"], fila["p_value"]),
    axis=1,
)

tabla_regresiones_problemas_hipotesis = (
    coeficientes_problemas_hipotesis
    .pivot_table(
        index="regresion",
        columns="mes",
        values="coeficiente_formateado",
        aggfunc="first",
    )
    .loc[[info["problema"] for info in problemas_hipotesis.values()]]
    .reset_index()
    .rename(columns={"regresion": "problema"})
)

tabla_regresiones_problemas_hipotesis.insert(
    1,
    "direccion_esperada",
    [info["direccion_esperada"] for info in problemas_hipotesis.values()],
)

tabla_regresiones_problemas_hipotesis


mes,problema,direccion_esperada,04/2025,05/2025,06/2025,07/2025
0,La vivienda,Baja tras el shock,0.0046,-0.0273**,0.0424***,0.0155
1,La inmigración,Sube tras el shock,-0.0160,-0.0336***,-0.0042,-0.0044
2,El paro,Sube tras el shock,-0.0171,-0.0115,-0.0505***,-0.0519***
3,El Gobierno y partidos o políticos/as concreto...,Sube tras el shock,-0.0373***,0.0102,-0.0327***,0.0079


Los resultados permiten matizar la hipótesis de forma bastante clara. La selección de los cuatro problemas sí está justificada: la vivienda aparece bastante más entre los no religiosos, mientras que inmigración, paro y Gobierno/partidos aparecen más entre los religiosos. Por tanto, tenía sentido comprobar si el shock movía la percepción de problemas en esa dirección.

Sin embargo, al pasar a las regresiones, la evidencia es bastante limitada. La vivienda sí baja en mayo respecto a marzo (`-0.0273**`), justo en el mes más directamente asociado al shock, lo que va en la dirección esperada. Pero este resultado no se mantiene después: en junio la vivienda aumenta (`0.0424***`) y en julio el efecto deja de ser claro.

Para los otros tres problemas la hipótesis no se confirma. La inmigración no sube tras el shock; de hecho, en mayo cae de forma significativa (`-0.0336***`). El paro tampoco aumenta, y en junio y julio cae con claridad. En el caso de Gobierno/partidos, tampoco aparece una subida consistente: abril y junio muestran coeficientes negativos, mientras que mayo y julio no presentan un cambio claro.

Por tanto, este bloque sugiere una conclusión importante para el trabajo: aunque religiosos y no religiosos priorizan problemas distintos, el shock religioso no parece trasladarse de forma fuerte y sistemática a la percepción de los principales problemas. La traslación hacia la agenda de problemas es mucho más débil y ésta debe estar más ligada a otras causas más relacionadas con los problemas en sí.


# 4. Religiosidad por ideología

Después de ver las regresiones principales, añado una comprobación que ayuda a interpretar mejor dónde aparece el cambio en religiosidad. Hasta ahora sabemos que en mayo aumenta la religiosidad declarada, pero todavía no sabemos si ese aumento se concentra en algún perfil ideológico concreto o si aparece de forma más repartida.

Para verlo, cruzo la ideología declarada, de 1 a 10, con la religiosidad en cada mes. La tabla muestra el porcentaje ponderado de personas religiosas dentro de cada punto de la escala ideológica. Así podemos observar si, tras el shock, la religiosidad aumenta sobre todo entre posiciones de izquierda, centro o derecha.


Primero construyo la tabla completa por valores de ideología. Mantengo solo las respuestas válidas entre 1 y 10 y calculo porcentajes ponderados con `Peso`. La última columna compara mayo con marzo, porque mayo es el mes en el que el shock religioso debería estar más presente.


In [28]:
df_ideologia = df_total[["Ideologia", "Mes/Año", "religiosidad_si_no", "Peso"]].dropna().copy()
df_ideologia = df_ideologia[df_ideologia["Ideologia"].between(1, 10)].copy()
df_ideologia["Ideologia"] = df_ideologia["Ideologia"].astype(int)

tabla_religiosidad_ideologia_mes = (
    df_ideologia
    .pivot_table(
        index="Ideologia",
        columns="Mes/Año",
        values="religiosidad_si_no",
        aggfunc=lambda serie: media_ponderada(serie, df_ideologia.loc[serie.index, "Peso"]),
    )
    .mul(100)
    .round(2)
)

tabla_religiosidad_ideologia_mes["Dif. mayo-marzo"] = (
    tabla_religiosidad_ideologia_mes["05/2025"]
    - tabla_religiosidad_ideologia_mes["03/2025"]
).round(2)

tabla_religiosidad_ideologia_mes


Mes/Año,03/2025,04/2025,05/2025,06/2025,07/2025,Dif. mayo-marzo
Ideologia,,,,,,
1,42.63,43.54,44.68,42.66,45.80,2.05
2,23.21,17.08,21.30,22.31,29.01,-1.91
3,31.05,28.50,29.57,30.08,29.25,-1.48
4,38.32,43.20,46.67,47.57,44.23,8.35
5,58.79,68.37,64.69,61.87,60.75,5.90
6,64.49,67.66,68.01,68.61,70.86,3.52
7,64.61,70.23,79.36,73.12,73.81,14.75
8,70.69,69.32,81.00,80.81,78.24,10.31
9,70.24,72.90,81.77,58.09,77.92,11.53


Para no sobreinterpretar variaciones pequeñas, reviso también el número de observaciones por ideología y mes. Esto es importante porque los extremos de la escala pueden tener menos casos y, por tanto, moverse más de un mes a otro.


In [29]:
tabla_n_ideologia_mes = (
    df_ideologia
    .pivot_table(
        index="Ideologia",
        columns="Mes/Año",
        values="religiosidad_si_no",
        aggfunc="size",
    )
)

tabla_n_ideologia_mes


Mes/Año,03/2025,04/2025,05/2025,06/2025,07/2025
Ideologia,,,,,
1,383,436,415,484,424
2,244,306,247,270,274
3,551,608,514,573,601
4,455,403,425,365,456
5,909,884,934,966,825
6,398,399,367,362,409
7,344,336,361,309,392
8,279,248,275,233,246
9,56,61,75,56,58


La tabla muestra que la relación entre ideología y religiosidad es muy marcada: a medida que nos movemos hacia posiciones más a la derecha, el porcentaje de personas religiosas aumenta de forma clara. Esto confirma algo que ya intuíamos por los gráficos anteriores, pero aquí lo vemos con más detalle dentro de cada punto de la escala ideológica.

Lo más interesante para el análisis del shock es la comparación entre marzo y mayo. El aumento de religiosidad no aparece de forma homogénea en toda la escala. En las posiciones 2 y 3, más cercanas a la izquierda, la religiosidad incluso baja ligeramente en mayo respecto a marzo. En cambio, el aumento es más claro en posiciones intermedias y de derecha moderada: por ejemplo, en la ideología 4 sube `8.35` puntos, en la 5 sube `5.90`, en la 7 sube `14.75` y en la 8 sube `10.31`.

Por tanto, esta tabla matiza la interpretación inicial. El shock no parece activar la religiosidad por igual entre todos los perfiles ideológicos, sino especialmente en zonas donde la religiosidad ya tenía una presencia relevante. Esto encaja con la idea de activación de una identidad previa: el episodio religioso no crea de cero una nueva religiosidad, sino que parece reforzar o hacer más visible una dimensión ya existente en determinados grupos.

También conviene ser prudente con los extremos de la escala, porque algunas categorías tienen menos observaciones, sobre todo la ideología 9. Por eso, más que interpretar cada casilla de forma aislada, lo más razonable es leer el patrón general: la subida de mayo aparece con más claridad en el centro y la derecha que en la izquierda.


# 5. Índice de intensidad religiosa

Hasta ahora he trabajado con una variable binaria: ser religioso o no serlo, entendiendo religiosidad como identificación católica, porque el shock analizado procede de la Iglesia católica. Esa variable es útil para ver si aumenta el número de personas que se declaran religiosas, pero deja fuera una cuestión interesante: puede que el shock no solo haga que algunas personas pasen a identificarse como religiosas, sino que también refuerce la intensidad religiosa de quienes ya lo eran.

Para aproximar esta idea construyo un índice sencillo que combina identificación religiosa y práctica. La lógica es que una persona no identificada como católica tenga valor 0; una persona católica no practicante tenga un nivel bajo; y una persona católica practicante, especialmente si asiste con frecuencia a oficios religiosos, tenga un nivel más alto. No es una medida perfecta de religiosidad, pero sí permite pasar de una comparación binaria a una escala gradual más informativa.


La escala queda definida entre 0 y 5. El valor 0 indica ausencia de identificación católica en esta medida. Entre las personas católicas, el índice suma dos elementos: la identificación declarada, diferenciando entre católico/a no practicante y católico/a practicante, y la frecuencia de práctica religiosa. De esta forma, una persona católica practicante y con asistencia frecuente obtiene un valor más alto que una persona católica no practicante que casi nunca asiste.


In [30]:
puntuacion_identidad_catolica = {
    "Católico/a no practicante": 1,
    "Católico/a practicante": 2,
}

puntuacion_practica = {
    "N.P.": 0,
    "Nunca": 0,
    "Casi nunca": 0,
    "Varias veces al año": 1,
    "Dos o tres veces al mes": 2,
    "Todos los domingos y festivos": 3,
    "Varias veces a la semana": 3,
}

df_total["indice_intensidad_religiosa"] = np.nan

sin_identificacion_catolica = (
    df_total["Religiosidad"].notna()
    & ~df_total["Religiosidad"].isin(puntuacion_identidad_catolica)
)
df_total.loc[sin_identificacion_catolica, "indice_intensidad_religiosa"] = 0

con_identificacion_catolica = df_total["Religiosidad"].isin(puntuacion_identidad_catolica)
con_practica_valida = df_total["Practica_religion"].isin(puntuacion_practica)
casos_indice_completo = con_identificacion_catolica & con_practica_valida

df_total.loc[casos_indice_completo, "indice_intensidad_religiosa"] = (
    df_total.loc[casos_indice_completo, "Religiosidad"].map(puntuacion_identidad_catolica)
    + df_total.loc[casos_indice_completo, "Practica_religion"].map(puntuacion_practica)
)

resumen_indice_intensidad = pd.DataFrame({
    "valor": [0, 1, 2, 3, 4, 5],
    "interpretacion": [
        "Sin identificación católica en esta medida",
        "Católico/a no practicante con práctica nula o muy baja",
        "Católico/a no practicante con algo de práctica o practicante con práctica baja",
        "Católico/a con práctica ocasional o media",
        "Católico/a practicante con práctica media o no practicante con práctica alta",
        "Católico/a practicante con práctica alta",
    ],
})

resumen_indice_intensidad


,valor,interpretacion
0,0,Sin identificación católica en esta medida
1,1,Católico/a no practicante con práctica nula o ...
2,2,Católico/a no practicante con algo de práctica...
3,3,Católico/a con práctica ocasional o media
4,4,Católico/a practicante con práctica media o no...
5,5,Católico/a practicante con práctica alta


Antes de usar el índice, reviso su distribución. Esto permite comprobar si la variable tiene suficientes observaciones en los distintos niveles y si la mayor parte de la muestra se concentra, como era esperable, en los valores bajos e intermedios.


In [31]:
tabla_distribucion_indice = (
    df_total
    .dropna(subset=["indice_intensidad_religiosa"])
    .groupby("indice_intensidad_religiosa")
    .apply(lambda grupo: pd.Series({
        "n": len(grupo),
        "porcentaje_ponderado": grupo["Peso"].sum() / df_total.dropna(subset=["indice_intensidad_religiosa"])["Peso"].sum() * 100,
    }))
    .reset_index()
    .rename(columns={"indice_intensidad_religiosa": "valor_indice"})
    .round(2)
)

tabla_distribucion_indice


,valor_indice,n,porcentaje_ponderado
0,0.0,9272.0,42.99
1,1.0,4893.0,26.62
2,2.0,1954.0,11.18
3,3.0,994.0,5.79
4,4.0,885.0,4.64
5,5.0,1724.0,8.77


Después calculo la evolución mensual de la media del índice. Si el shock no solo aumenta la religiosidad declarada, sino también la intensidad religiosa, esperaríamos que la media subiera especialmente en mayo respecto a marzo. Para facilitar la lectura, añado la variación porcentual relativa entre mayo y marzo.


In [32]:
df_indice = df_total.dropna(subset=["indice_intensidad_religiosa", "Peso"]).copy()

serie_intensidad_total_mes = (
    df_indice
    .groupby("Mes/Año")
    .apply(lambda grupo: media_ponderada(grupo["indice_intensidad_religiosa"], grupo["Peso"]))
)

tabla_intensidad_total_mes = serie_intensidad_total_mes.to_frame("media_indice").T.round(3)

variacion_total_mayo_marzo = (
    (serie_intensidad_total_mes["05/2025"] - serie_intensidad_total_mes["03/2025"])
    / serie_intensidad_total_mes["03/2025"]
    * 100
)
tabla_intensidad_total_mes["Var. mayo-marzo"] = f"{variacion_total_mayo_marzo:+.2f}%"

tabla_intensidad_total_mes


Mes/Año,03/2025,04/2025,05/2025,06/2025,07/2025,Var. mayo-marzo
media_indice,1.287,1.311,1.317,1.218,1.306,+2.40%


Por último, miro esta misma media por partido político. Esta tabla ayuda a comprobar si el posible aumento de intensidad religiosa aparece en todos los electorados o si se concentra en alguno concreto. Añado también una fila de total de la muestra y la variación porcentual relativa entre mayo y marzo.


In [33]:
partidos_interes = ["PSOE", "PP", "VOX", "Sumar"]

tabla_intensidad_partido_mes = (
    df_indice[df_indice["Intencion_voto"].isin(partidos_interes)]
    .pivot_table(
        index="Intencion_voto",
        columns="Mes/Año",
        values="indice_intensidad_religiosa",
        aggfunc=lambda serie: media_ponderada(serie, df_indice.loc[serie.index, "Peso"]),
    )
    .reindex(partidos_interes)
)

tabla_intensidad_total_fila = (
    df_indice
    .groupby("Mes/Año")
    .apply(lambda grupo: media_ponderada(grupo["indice_intensidad_religiosa"], grupo["Peso"]))
    .to_frame()
    .T
)
tabla_intensidad_total_fila.index = ["Total"]

tabla_intensidad_partido_mes = pd.concat([
    tabla_intensidad_partido_mes,
    tabla_intensidad_total_fila,
])

variacion_intensidad_partido = (
    (tabla_intensidad_partido_mes["05/2025"] - tabla_intensidad_partido_mes["03/2025"])
    / tabla_intensidad_partido_mes["03/2025"]
    * 100
)

tabla_intensidad_partido_mes = tabla_intensidad_partido_mes.round(3)
tabla_intensidad_partido_mes["Var. mayo-marzo"] = variacion_intensidad_partido.map(lambda valor: f"{valor:+.2f}%")

tabla_intensidad_partido_mes


Mes/Año,03/2025,04/2025,05/2025,06/2025,07/2025,Var. mayo-marzo
PSOE,0.958,0.933,0.985,0.921,1.002,+2.84%
PP,2.049,2.064,2.124,1.884,2.109,+3.66%
VOX,1.414,1.770,1.394,1.599,1.616,-1.41%
Sumar,0.330,0.437,0.307,0.499,0.458,-6.81%
Total,1.287,1.311,1.317,1.218,1.306,+2.40%


Como comprobación final, estimo una regresión sencilla con el índice como variable dependiente. Mantengo los mismos controles que en las regresiones base. Esta regresión no sustituye a la tabla, pero permite ver si el cambio mensual se mantiene al controlar por edad, sexo y estudios.


In [34]:
variables_regresion_indice = [
    "indice_intensidad_religiosa",
    "Mes/Año",
    "Edad",
    "Sexo",
    "Estudios",
    "Peso",
]

df_reg_indice = df_total[variables_regresion_indice].dropna().copy()
modelo_reg_indice = estimar_wls_base(df_reg_indice, "indice_intensidad_religiosa")

coeficientes_indice = extraer_coeficientes_meses(
    modelo_reg_indice,
    "Índice de intensidad religiosa",
    "indice_intensidad_religiosa",
)

coeficientes_indice["coeficiente_formateado"] = coeficientes_indice.apply(
    lambda fila: formatear_coeficiente(fila["coeficiente"], fila["p_value"]),
    axis=1,
)

tabla_regresion_indice = (
    coeficientes_indice
    .pivot_table(
        index="regresion",
        columns="mes",
        values="coeficiente_formateado",
        aggfunc="first",
    )
    .reset_index()
)

tabla_regresion_indice


mes,regresion,04/2025,05/2025,06/2025,07/2025
0,Índice de intensidad religiosa,0.0168,0.0379,-0.0692,0.0142


El índice permite comprobar una hipótesis más exigente que la variable binaria de religiosidad. Ya no preguntamos solo si aumenta el porcentaje de personas que se declaran católicas, sino si aumenta la intensidad media de esa religiosidad.

El resultado es bastante moderado. En el total de la muestra, la media del índice pasa de `1.287` en marzo a `1.317` en mayo, lo que supone una variación relativa de `+2.40%`. La dirección va en el sentido esperado, pero el cambio es pequeño. Además, en la regresión con controles el coeficiente de mayo es positivo (`0.0379`), pero no aparece como estadísticamente significativo.

Por partidos, la subida relativa de mayo respecto a marzo se observa sobre todo en el PP (`+3.66%`) y, en menor medida, en el PSOE (`+2.84%`). En cambio, en VOX (`-1.41%`) y Sumar (`-6.81%`) la media del índice no aumenta en mayo respecto a marzo. Esto indica que la hipótesis de una intensificación religiosa general no se confirma de forma clara.

La lectura más razonable es que el shock tiene un efecto más visible sobre la identificación religiosa declarada que sobre la intensidad religiosa entendida como combinación de identidad y práctica. Es decir, los datos son más compatibles con una activación simbólica o identitaria de la religión que con un cambio fuerte en el grado de religiosidad efectiva de la población. Esta diferencia es importante para el argumento del TFG, porque refuerza la idea de que el shock actúa primero sobre la saliencia y la identidad, mientras que sus efectos sobre comportamientos o intensidades más estables son más limitados.


# 6. Tablas completas de regresión

Para preparar la redacción final del TFG, construyo ahora tablas de regresión más completas, pero manteniendo la misma estructura del capítulo. Es decir, no junto todos los modelos en una única tabla, sino que genero una tabla para cada bloque del análisis: religiosidad, voto, electorados, problemas e intensidad religiosa.


Primero defino una función común para que todas las tablas tengan el mismo formato. Los coeficientes aparecen en la unidad original de la variable dependiente. En las variables binarias, un coeficiente de `0.0410` equivale a 4,10 puntos porcentuales. Las filas finales recogen solo observaciones, $R^2$ y media de la variable dependiente; controles, pesos y errores robustos se explican en la nota de cada tabla para no repetir información.


In [35]:
OUTPUT_TABLAS_LATEX = Path("Outputs/tablas_latex")
OUTPUT_TABLAS_LATEX.mkdir(parents=True, exist_ok=True)

TERMINOS_MESES = {
    "Abril 2025": 'C(Q("Mes/Año"))[T.04/2025]',
    "Mayo 2025": 'C(Q("Mes/Año"))[T.05/2025]',
    "Junio 2025": 'C(Q("Mes/Año"))[T.06/2025]',
    "Julio 2025": 'C(Q("Mes/Año"))[T.07/2025]',
}


def formatear_coeficiente_tabla(modelo, termino, decimales=4):
    if termino not in modelo.params.index:
        return ""
    coeficiente = modelo.params[termino]
    p_value = modelo.pvalues[termino]
    return f"{coeficiente:.{decimales}f}{estrellas_significatividad(p_value)}"


def formatear_error_estandar_tabla(modelo, termino, decimales=4):
    if termino not in modelo.bse.index:
        return ""
    return f"({modelo.bse[termino]:.{decimales}f})"


def media_dependiente_ponderada(df_modelo, variable_dependiente, peso="Peso"):
    return media_ponderada(df_modelo[variable_dependiente], df_modelo[peso])


def crear_tabla_regresion_completa(modelos_info, terminos=TERMINOS_MESES, decimales=4):
    filas = []

    for etiqueta_termino, termino in terminos.items():
        fila_coeficientes = {"Variable": etiqueta_termino}
        fila_errores = {"Variable": ""}

        for info in modelos_info:
            columna = info["columna"]
            modelo = info["modelo"]
            fila_coeficientes[columna] = formatear_coeficiente_tabla(modelo, termino, decimales)
            fila_errores[columna] = formatear_error_estandar_tabla(modelo, termino, decimales)

        filas.extend([fila_coeficientes, fila_errores])

    filas_resumen = [
        {
            "Variable": "Observaciones",
            **{info["columna"]: f"{int(info['modelo'].nobs):,}".replace(",", ".") for info in modelos_info},
        },
        {
            "Variable": "$R^2$",
            **{info["columna"]: f"{info['modelo'].rsquared:.3f}" for info in modelos_info},
        },
        {
            "Variable": "Media dep.",
            **{
                info["columna"]: f"{media_dependiente_ponderada(info['df'], info['variable_dependiente']):.3f}"
                for info in modelos_info
            },
        },
    ]
    filas.extend(filas_resumen)

    tabla = pd.DataFrame(filas).set_index("Variable")
    tabla.index.name = ""
    return tabla


def guardar_tabla_latex(tabla, nombre_archivo, caption, label, nota=None):
    ruta = OUTPUT_TABLAS_LATEX / nombre_archivo
    columnas = list(tabla.columns)
    formato_columnas = "l" + "c" * len(columnas)
    bs = chr(92)
    salto = chr(10)

    if nota is None:
        nota = (
            "Nota: marzo de 2025 es la categoría de referencia. "
            "Los errores estándar robustos HC1 aparecen entre paréntesis. "
            "Todas las regresiones incluyen controles por edad, sexo y nivel de estudios, "
            "y utilizan la ponderación muestral del CIS. "
            "$* p<0.1$, $** p<0.05$, $*** p<0.01$."
        )

    lineas = [
        f"{bs}begin{{table}}[H]",
        f"{bs}centering",
        f"{bs}caption{{{caption}}}",
        f"{bs}label{{{label}}}",
        f"{bs}small",
        f"{bs}begin{{tabular}}{{{formato_columnas}}}",
        f"{bs}hline",
        " & " + " & ".join(columnas) + f" {bs}{bs}",
        f"{bs}hline",
    ]

    for indice, fila in tabla.iterrows():
        valores = [str(fila[columna]) for columna in columnas]
        lineas.append(str(indice) + " & " + " & ".join(valores) + f" {bs}{bs}")

    lineas.extend([
        f"{bs}hline",
        f"{bs}end{{tabular}}",
        f"{bs}begin{{flushleft}}",
        f"{bs}footnotesize {nota}",
        f"{bs}end{{flushleft}}",
        f"{bs}end{{table}}",
        "",
    ])

    ruta.write_text(salto.join(lineas), encoding="utf-8")
    return ruta


La primera tabla completa corresponde al primer bloque del capítulo: religiosidad declarada y práctica religiosa.


In [36]:
modelos_religiosidad_practica = [
    {"columna": "Religiosidad", "modelo": modelo_reg1, "df": df_reg1, "variable_dependiente": "religiosidad_si_no"},
    {"columna": "Práctica", "modelo": modelo_reg2, "df": df_reg2, "variable_dependiente": "practica_religion_si_no"},
]

tabla_reg_religiosidad_practica = crear_tabla_regresion_completa(modelos_religiosidad_practica)
ruta_tabla_religiosidad_practica = guardar_tabla_latex(
    tabla_reg_religiosidad_practica,
    "tabla_reg_religiosidad_practica.tex",
    "Efecto mensual sobre religiosidad declarada y práctica religiosa",
    "tab:religiosidad-practica-completa",
)

display(tabla_reg_religiosidad_practica)
print(f"Tabla guardada en: {ruta_tabla_religiosidad_practica}")


,Religiosidad,Práctica
,,
Abril 2025,0.0116,0.0112
,(0.0148),(0.0145)
Mayo 2025,0.0410***,0.0271*
,(0.0146),(0.0144)
Junio 2025,0.0168,-0.0115
,(0.0150),(0.0144)
Julio 2025,0.0187,-0.0023
,(0.0155),(0.0151)
Observaciones,20.057,19.922


Tabla guardada en: Outputs/tablas_latex/tabla_reg_religiosidad_practica.tex


La segunda tabla recoge las regresiones de intención de voto a los cuatro partidos principales.


In [37]:
modelos_voto_completos = [
    {"columna": "PSOE", "modelo": modelo_reg3, "df": df_reg3, "variable_dependiente": "voto_psoe_si_no"},
    {"columna": "PP", "modelo": modelo_reg4, "df": df_reg4, "variable_dependiente": "voto_pp_si_no"},
    {"columna": "VOX", "modelo": modelo_reg5, "df": df_reg5, "variable_dependiente": "voto_vox_si_no"},
    {"columna": "Sumar", "modelo": modelo_reg6, "df": df_reg6, "variable_dependiente": "voto_sumar_si_no"},
]

tabla_reg_voto = crear_tabla_regresion_completa(modelos_voto_completos)
ruta_tabla_voto = guardar_tabla_latex(
    tabla_reg_voto,
    "tabla_reg_voto.tex",
    "Efecto mensual sobre intención de voto",
    "tab:voto-completa",
)

display(tabla_reg_voto)
print(f"Tabla guardada en: {ruta_tabla_voto}")


,PSOE,PP,VOX,Sumar
,,,,
Abril 2025,0.0034,-0.0038,0.0247**,-0.0067
,(0.0138),(0.0122),(0.0107),(0.0057)
Mayo 2025,-0.0078,0.0141,0.0118,-0.0053
,(0.0134),(0.0122),(0.0098),(0.0060)
Junio 2025,0.0046,-0.0077,0.0036,-0.0043
,(0.0140),(0.0121),(0.0096),(0.0058)
Julio 2025,-0.0541***,-0.0020,0.0528***,0.0040
,(0.0132),(0.0126),(0.0117),(0.0067)
Observaciones,19.450,19.450,19.450,19.450


Tabla guardada en: Outputs/tablas_latex/tabla_reg_voto.tex


La tercera tabla muestra las regresiones separadas por electorado. En todos los casos la variable dependiente es la religiosidad declarada, pero la muestra se restringe a quienes declaran intención de voto a cada partido.


In [38]:
modelos_electorados_completos = [
    {
        "columna": partido,
        "modelo": modelos_religiosidad_electorados[partido]["modelo"],
        "df": modelos_religiosidad_electorados[partido]["df"],
        "variable_dependiente": "religiosidad_si_no",
    }
    for partido in partidos_interes
]

tabla_reg_electorados = crear_tabla_regresion_completa(modelos_electorados_completos)
ruta_tabla_electorados = guardar_tabla_latex(
    tabla_reg_electorados,
    "tabla_reg_electorados.tex",
    "Regresiones de religiosidad por electorado",
    "tab:electorados-completa",
)

display(tabla_reg_electorados)
print(f"Tabla guardada en: {ruta_tabla_electorados}")


,PSOE,PP,VOX,Sumar
,,,,
Abril 2025,-0.0204,-0.0150,0.0678,-0.0137
,(0.0312),(0.0265),(0.0519),(0.0474)
Mayo 2025,0.0299,0.0543**,0.0268,0.0129
,(0.0305),(0.0238),(0.0527),(0.0522)
Junio 2025,0.0098,-0.0147,0.0246,0.0302
,(0.0318),(0.0273),(0.0522),(0.0497)
Julio 2025,-0.0108,0.0054,0.0336,0.0900*
,(0.0326),(0.0269),(0.0528),(0.0544)
Observaciones,4.437,4.276,1.736,1.130


Tabla guardada en: Outputs/tablas_latex/tabla_reg_electorados.tex


La cuarta tabla recoge las regresiones sobre percepción de problemas. Cada columna utiliza como variable dependiente una dummy que indica si la persona menciona ese problema entre sus tres respuestas.


In [39]:
nombres_cortos_problemas = {
    "problema_vivienda": "Vivienda",
    "problema_inmigracion": "Inmigración",
    "problema_paro": "Paro",
    "problema_gobierno_partidos": "Gobierno/partidos",
}

modelos_problemas_completos = [
    {
        "columna": nombres_cortos_problemas[variable],
        "modelo": resultado["modelo"],
        "df": resultado["df"],
        "variable_dependiente": variable,
    }
    for variable, resultado in modelos_problemas_hipotesis.items()
]

tabla_reg_problemas = crear_tabla_regresion_completa(modelos_problemas_completos)
ruta_tabla_problemas = guardar_tabla_latex(
    tabla_reg_problemas,
    "tabla_reg_problemas.tex",
    "Efecto mensual sobre la percepción de problemas",
    "tab:problemas-completa",
)

display(tabla_reg_problemas)
print(f"Tabla guardada en: {ruta_tabla_problemas}")


,Vivienda,Inmigración,Paro,Gobierno/partidos
,,,,
Abril 2025,0.0046,-0.0160,-0.0171,-0.0373***
,(0.0129),(0.0128),(0.0128),(0.0119)
Mayo 2025,-0.0273**,-0.0336***,-0.0115,0.0102
,(0.0126),(0.0121),(0.0126),(0.0122)
Junio 2025,0.0424***,-0.0042,-0.0505***,-0.0327***
,(0.0136),(0.0128),(0.0123),(0.0115)
Julio 2025,0.0155,-0.0044,-0.0519***,0.0079
,(0.0138),(0.0133),(0.0128),(0.0132)
Observaciones,20.057,20.057,20.057,20.057


Tabla guardada en: Outputs/tablas_latex/tabla_reg_problemas.tex


Por último, genero la tabla completa de la regresión del índice de intensidad religiosa. En este caso la variable dependiente no es binaria, sino el índice entre 0 y 5 construido anteriormente.


In [40]:
modelos_intensidad_completos = [
    {
        "columna": "Índice intensidad",
        "modelo": modelo_reg_indice,
        "df": df_reg_indice,
        "variable_dependiente": "indice_intensidad_religiosa",
    }
]

tabla_reg_intensidad = crear_tabla_regresion_completa(modelos_intensidad_completos)
ruta_tabla_intensidad = guardar_tabla_latex(
    tabla_reg_intensidad,
    "tabla_reg_intensidad.tex",
    "Regresión sobre el índice de intensidad religiosa",
    "tab:intensidad-completa",
)

display(tabla_reg_intensidad)
print(f"Tabla guardada en: {ruta_tabla_intensidad}")


,Índice intensidad
,
Abril 2025,0.0168
,(0.0502)
Mayo 2025,0.0379
,(0.0476)
Junio 2025,-0.0692
,(0.0474)
Julio 2025,0.0142
,(0.0516)
Observaciones,19.708


Tabla guardada en: Outputs/tablas_latex/tabla_reg_intensidad.tex


# 7. Síntesis final del análisis principal

Con estos bloques queda cerrado el análisis principal del trabajo. La idea era seguir una secuencia: primero comprobar si el shock se refleja en la dimensión religiosa, después ver si esa activación aparece de forma distinta según electorados e ideología y, por último, analizar si se traslada también al voto y a la percepción de problemas

| Bloque | Resultado principal | Lectura para el TFG |
|---|---|---|
| Regresiones base | La religiosidad declarada aumenta en mayo (`4.10%`) y la práctica religiosa también sube, aunque con menor fuerza (`2.71%`). | Es la evidencia más clara de que el shock religioso tuvo algún efecto observable. |
| Intención de voto e ideología | Los cambios en voto son más parciales: destaca la caída del PSOE en julio y el aumento de VOX en abril y julio, pero no aparece una traslación política limpia en mayo. | La relación entre shock religioso y voto existe solo de forma indirecta y mezclada con otros acontecimientos políticos. |
| Composición religiosa de electorados | En mayo aumenta la religiosidad dentro del electorado del PP (`5.43%`) y en julio aparece una subida en Sumar (`9.00%`). | El shock no afecta igual a todos los electorados; parece activar perfiles concretos más que producir un cambio general. |
| Percepción de problemas | La vivienda cae en mayo (`2.73%`), pero inmigración, paro y Gobierno/partidos no suben de forma consistente. | La traslación del shock a la agenda de problemas es débil. |
| Religiosidad por ideología | El aumento de mayo se concentra más en posiciones de centro y derecha moderada que en la izquierda. | El shock parece activar una identidad religiosa previa, no crear un cambio homogéneo en toda la población. |
| Índice de intensidad religiosa | La media del índice sube muy poco en mayo, de `1.287` a `1.317`, y la regresión no es significativa. | El efecto es más identitario o simbólico que una intensificación fuerte de la práctica o del grado de religiosidad. |

En conjunto, los resultados apoyan una interpretación prudente. El shock religioso sí parece aumentar la religiosidad declarada en el momento de mayor saliencia, especialmente en mayo. Sin embargo, ese efecto se debilita cuando pasamos a dimensiones más alejadas del shock: la práctica religiosa, el voto, la percepción de problemas y la intensidad religiosa muestran resultados más parciales o menos consistentes.

Por tanto, la conclusión del análisis no debería ser que el shock transformó de manera clara el comportamiento político de la ciudadanía española. La lectura más defendible es que el shock activó temporalmente la dimensión religiosa, sobre todo como identidad declarada, y que esa activación tuvo una proyección limitada y desigual sobre las preferencias políticas y la percepción de problemas. Esta conclusión encaja con el marco teórico del trabajo: la religión puede seguir siendo una identidad relevante en una sociedad secularizada, pero su efecto aparece de forma condicionada, parcial y dependiente del contexto.
